# 04 — Operations Analysis

## tl;dr

This notebook turns demand evidence into operational diagnostics: stable/volatile family segments, store performance profiles, promotion dependence indicators, and peak-demand windows. Recommendations are framed as planning actions, not precise inventory optimization.

## Context & Methods

Demand variability is measured on daily aggregates using coefficient of variation (CV = standard deviation / mean). Median demand and median CV create transparent four-quadrant family segments. These are relative portfolio segments, not universal industry thresholds.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
# Resolve the repository root dynamically so the notebook works from either
# the project root or the notebooks directory without a local absolute path.
candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (candidate, *candidate.parents)
    if (path / "data").is_dir() and (path / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DATA_DIR = PROJECT_ROOT / "outputs" / "processed_data"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
for directory in (PROCESSED_DATA_DIR, OUTPUT_DATA_DIR, FIGURE_DIR, TABLE_DIR):
    # Creating output folders is safe; raw source files are never modified.
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from src.data_loader import load_all_available
from src.preprocessing import build_analysis_panel
from src.visualization import PROJECT_COLORS, save_figure, set_project_style

set_project_style()
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")
datasets = load_all_available(RAW_DATA_DIR)
panel = build_analysis_panel(datasets)

## Results

### 1. Family demand variability and operational segments

**Why this step is needed:** Combining demand level with coefficient of variation produces practical portfolio groups: routine high-volume flows, uncertain high-volume flows, and lower-priority stable or volatile categories.

In [ ]:
family_daily = panel.groupby(["family", "date"], observed=True)["sales"].sum().reset_index()
family_ops = family_daily.groupby("family", observed=True)["sales"].agg(
    mean_daily_demand="mean", std_daily_demand="std", total_sales="sum"
)
family_ops["cv"] = family_ops["std_daily_demand"] / family_ops["mean_daily_demand"]
demand_cut = family_ops["mean_daily_demand"].median()
cv_cut = family_ops["cv"].median()
family_ops["demand_level"] = np.where(family_ops["mean_daily_demand"].ge(demand_cut), "High demand", "Low demand")
family_ops["variability_level"] = np.where(family_ops["cv"].ge(cv_cut), "Volatile", "Stable")
family_ops["segment"] = family_ops["demand_level"] + " / " + family_ops["variability_level"]
display(family_ops.sort_values(["segment", "total_sales"], ascending=[True, False]))

fig, ax = plt.subplots(figsize=(11, 7))
segment_colors = {
    "High demand / Stable": PROJECT_COLORS[1], "High demand / Volatile": PROJECT_COLORS[4],
    "Low demand / Stable": PROJECT_COLORS[0], "Low demand / Volatile": PROJECT_COLORS[3],
}
for segment, values in family_ops.groupby("segment"):
    ax.scatter(values["mean_daily_demand"], values["cv"], label=segment, s=65, alpha=.8, color=segment_colors[segment])
ax.axvline(demand_cut, color="grey", linestyle="--")
ax.axhline(cv_cut, color="grey", linestyle="--")
for family in family_ops.nlargest(6, "total_sales").index:
    ax.annotate(family, (family_ops.loc[family, "mean_daily_demand"], family_ops.loc[family, "cv"]), fontsize=8)
ax.set(title="Product-family demand segmentation", xlabel="Mean daily demand", ylabel="Coefficient of variation")
ax.legend()
save_figure(fig, FIGURE_DIR, "operations_01_family_segments.png")
plt.show()

### 2. Store performance and operating profiles

**Why this step is needed:** Store planning should reflect scale, variability, traffic, promotion exposure, and lifecycle coverage. The 90% coverage rule prevents later openings from being mislabeled as ordinary volatility.

In [ ]:
store_daily = panel.groupby(["store_nbr", "date"], observed=True).agg(
    sales=("sales", "sum"), transactions=("transactions", "max"), onpromotion=("onpromotion", "sum")
).reset_index()
store_ops = store_daily.groupby("store_nbr").agg(
    average_daily_sales=("sales", "mean"),
    std_daily_sales=("sales", "std"),
    average_daily_transactions=("transactions", "mean"),
    transaction_coverage_pct=("transactions", lambda values: 100 * values.notna().mean()),
    promoted_item_days=("onpromotion", lambda values: values.gt(0).sum()),
    active_sales_days=("sales", lambda values: values.gt(0).sum()),
    observed_days=("date", "size"),
)
store_ops["demand_cv"] = store_ops["std_daily_sales"] / store_ops["average_daily_sales"]
store_ops["promotion_active_days_pct"] = 100 * store_ops["promoted_item_days"] / store_ops["observed_days"]
store_ops["active_sales_days_pct"] = 100 * store_ops["active_sales_days"] / store_ops["observed_days"]
store_ops["comparable_store"] = (
    store_ops["transaction_coverage_pct"].ge(90)
    & store_ops["active_sales_days_pct"].ge(90)
)

monthly_store = store_daily.assign(month=store_daily["date"].dt.month).groupby(["store_nbr", "month"])["sales"].mean()
store_ops["seasonal_sensitivity"] = monthly_store.groupby("store_nbr").max() / monthly_store.groupby("store_nbr").mean()
store_ops = store_ops.join(datasets["stores.csv"].set_index("store_nbr"))
display(store_ops.sort_values("average_daily_sales", ascending=False).head(15))
display(store_ops.loc[~store_ops["comparable_store"]].sort_values("transaction_coverage_pct"))

fig, ax = plt.subplots(figsize=(11, 7))
comparable_ops = store_ops.loc[store_ops["comparable_store"]]
scatter = ax.scatter(
    comparable_ops["average_daily_sales"], comparable_ops["demand_cv"],
    s=20 + 2 * comparable_ops["average_daily_transactions"].fillna(0) ** .5,
    c=comparable_ops["promotion_active_days_pct"], cmap="viridis", alpha=.8
)
ax.set(title="Store operating profiles — comparable mature stores", xlabel="Average daily sales", ylabel="Demand CV")
fig.colorbar(scatter, ax=ax, label="Promotion-active days (%)")
save_figure(fig, FIGURE_DIR, "operations_02_store_profiles.png")
plt.show()

### 3. Promotion planning indicators

**Why this step is needed:** Promotion-associated demand differs by family, so category-specific ratios can prioritize staffing and replenishment coordination while avoiding unsupported causal-lift claims.

In [ ]:
promo = panel.assign(promoted=panel["onpromotion"].gt(0)).groupby(
    ["family", "promoted"], observed=True
)["sales"].agg(["mean", "median", "count"]).unstack()
promo_metrics = pd.DataFrame(index=promo.index)
promo_metrics["mean_not_promoted"] = promo[("mean", False)]
promo_metrics["mean_promoted"] = promo[("mean", True)]
promo_metrics["observed_ratio"] = promo_metrics["mean_promoted"] / promo_metrics["mean_not_promoted"].replace(0, np.nan)
promo_metrics["promoted_observations"] = promo[("count", True)]
promo_metrics = promo_metrics.join(family_ops[["segment", "total_sales"]])
display(promo_metrics.sort_values("observed_ratio", ascending=False).head(15))

fig, ax = plt.subplots(figsize=(11, 7))
promo_metrics.nlargest(15, "total_sales").sort_values("observed_ratio")["observed_ratio"].plot(
    kind="barh", ax=ax, color=PROJECT_COLORS[3]
)
ax.axvline(1, color="black", linestyle="--", linewidth=1)
ax.set(title="Promotion association among high-volume families", xlabel="Promoted/non-promoted mean-sales ratio", ylabel="Family")
save_figure(fig, FIGURE_DIR, "operations_03_promotion_planning.png")
plt.show()

### 4. Peak-demand calendar for staffing and capacity

**Why this step is needed:** Recurring weekday and monthly peaks translate demand history into concrete reviews of labor, checkout, receiving, shelf replenishment, and supplier capacity.

In [ ]:
network_daily = panel.groupby("date", observed=True).agg(
    sales=("sales", "sum"), is_holiday=("is_holiday", "max"), onpromotion=("onpromotion", "sum")
).reset_index()
network_daily["weekday"] = network_daily["date"].dt.day_name()
network_daily["month"] = network_daily["date"].dt.month
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

peak_weekdays = network_daily.groupby("weekday")["sales"].agg(["mean", "std"]).reindex(weekday_order)
peak_months = network_daily.groupby("month")["sales"].agg(["mean", "std"])
holiday_peak = network_daily.groupby("is_holiday")["sales"].agg(["mean", "median", "std", "count"])
promotion_bins = pd.qcut(network_daily["onpromotion"], 4, duplicates="drop")
promotion_peak = network_daily.groupby(promotion_bins, observed=True)["sales"].agg(["mean", "median", "count"])
display(peak_weekdays.sort_values("mean", ascending=False))
display(peak_months.sort_values("mean", ascending=False))
display(holiday_peak)
display(promotion_peak)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
peak_weekdays["mean"].plot(kind="bar", ax=axes[0], color=PROJECT_COLORS[0])
axes[0].set(title="Average network demand by weekday", xlabel="Weekday", ylabel="Average daily sales")
peak_months["mean"].plot(kind="bar", ax=axes[1], color=PROJECT_COLORS[1])
axes[1].set(title="Average network demand by month", xlabel="Month", ylabel="Average daily sales")
fig.tight_layout()
save_figure(fig, FIGURE_DIR, "operations_04_peak_calendar.png")
plt.show()

### 5. Save operational evidence tables

**Why this step is needed:** The final insight notebook should read small, reviewed evidence tables rather than repeat heavy transformations or depend on hidden notebook state.

In [ ]:
family_ops.reset_index().to_csv(TABLE_DIR / "family_operations_segments.csv", index=False)
store_ops.reset_index().to_csv(TABLE_DIR / "store_operations_summary.csv", index=False)
promo_metrics.reset_index().to_csv(TABLE_DIR / "promotion_operations_summary.csv", index=False)
peak_weekdays.reset_index().to_csv(TABLE_DIR / "peak_weekdays.csv", index=False)
peak_months.reset_index().to_csv(TABLE_DIR / "peak_months.csv", index=False)
print("Saved five operations evidence tables.")

## Takeaways — implications and possible actions

In [ ]:
segment_counts = family_ops["segment"].value_counts()
peak_day = peak_weekdays["mean"].idxmax()
peak_month = int(peak_months["mean"].idxmax())
most_variable_store = int(store_ops.loc[store_ops["comparable_store"], "demand_cv"].idxmax())
display(Markdown(f'''
- **Finding:** Family segments contain {segment_counts.to_dict()}. **Implication:** demand level and variability require different controls. **Possible action:** use routine replenishment for stable families and shorter review cycles or buffers for volatile families, subject to lead-time and stock data.
- **Finding:** Store {most_variable_store} has the highest daily-demand CV among stores meeting the 90% coverage rule. **Implication:** network-wide averages may understate local planning uncertainty. **Possible action:** use store-specific staffing and replenishment tolerances; treat later-opening stores separately.
- **Finding:** The highest average network demand occurs on **{peak_day}** and in month **{peak_month}**. **Implication:** capacity requirements are not uniform. **Possible action:** review labor rosters, receiving capacity, and supplier schedules ahead of recurring peaks.
- **Finding:** Promotion-associated sales ratios vary across families. **Implication:** a single promotion rule is unlikely to fit the portfolio. **Possible action:** prioritize controlled evaluation by category and store rather than interpreting descriptive ratios as causal lift.
- **Limitation:** Exact safety stocks, reorder points, or service-level gains cannot be calculated without inventory positions, stockouts, lead times, costs, and service targets.
'''))